In [1]:
import os
print("reading from:", os.environ.get("LCT_TABLES"))

reading from: D:\FDR-Corelation\projects\fdr-correlations\paper_results\tables


In [2]:
from pathlib import Path
import pandas as pd, numpy as np, re, matplotlib.pyplot as plt
import os

TABLES = Path(os.environ.get("LCT_TABLES", "../results/tables"))
files = sorted(TABLES.glob("robust_*_p*.csv"))
print(f"found {len(files)} robustness files")
df = pd.concat([pd.read_csv(f).assign(file=f.name) for f in files], ignore_index=True) if files else pd.DataFrame()
print("columns present:", [c for c in ["cov_kind","n1","n2","rho","seed","model","p"] if c in df.columns])
df.head()

found 18 robustness files
columns present: ['cov_kind', 'n1', 'n2', 'rho', 'seed', 'model', 'p']


,model,x_model,p,n1,n2,rho,block,cov_kind,decay,seed,...,fdr_lctb_0.05_B100,power_lctb_0.05_B100,t_lctb_0.1_B100,R_lctb_0.1_B100,V_lctb_0.1_B100,S_lctb_0.1_B100,fdr_lctb_0.1_B100,power_lctb_0.1_B100,wall_time_s,file
0,gaussian,gaussian,250,120,120,0.6,20,block_ar1,0.6,0,...,0.000000,0.110526,4.342627,21,0,21,0.000000,0.110526,1.892950,robust_gaussian_p250_covblock_ar1_n120_120_rho...
1,gaussian,gaussian,250,120,120,0.6,20,block_ar1,0.6,1,...,0.041667,0.121053,4.010206,25,2,23,0.080000,0.121053,1.727668,robust_gaussian_p250_covblock_ar1_n120_120_rho...
2,gaussian,gaussian,250,120,120,0.6,20,block_ar1,0.6,2,...,0.000000,0.100000,4.265054,20,0,20,0.000000,0.105263,1.591634,robust_gaussian_p250_covblock_ar1_n120_120_rho...
3,gaussian,gaussian,250,120,120,0.6,20,block_ar1,0.6,3,...,0.043478,0.115789,3.968770,29,4,25,0.137931,0.131579,1.900261,robust_gaussian_p250_covblock_ar1_n120_120_rho...
4,gaussian,gaussian,250,120,120,0.6,20,block_ar1,0.6,4,...,0.000000,0.110526,4.079401,23,2,21,0.086957,0.110526,1.778264,robust_gaussian_p250_covblock_ar1_n120_120_rho...


In [3]:
def collect(df, prefix, yname):
    cols = [c for c in df.columns if isinstance(c, str) and c.startswith(prefix)]
    if not cols: return pd.DataFrame()
    L = df.melt(id_vars=[c for c in df.columns if c not in cols], value_vars=cols,
                var_name="metric", value_name=yname)
    L["alpha"] = L["metric"].str.extract(r'_(0\.05|0\.10)')
    L["vm"]    = L["metric"].str.extract(r'^.*_lct_([a-z_]+)_')
    L["method"]= np.where(L["metric"].str.contains("_bh_"), "BH",
                  np.where(L["metric"].str.contains("_by_"), "BY",
                  np.where(L["metric"].str.contains("_lctb_"), "LCT-B",
                  np.where(L["metric"].str.contains("_lct_"), "LCT-N","?"))))
    return L

pwr = pd.concat([collect(df, p, "power") for p in ["power_bh_","power_by_","power_lct_","power_lctb_"]], ignore_index=True)
fdr = pd.concat([collect(df, p, "fdr")   for p in ["fdr_bh_","fdr_by_","fdr_lct_","fdr_lctb_"]],   ignore_index=True)

KEYS = [k for k in ["cov_kind","n1","n2","rho","alpha","method","vm"] if k in fdr.columns]
print("grouping by:", KEYS)

agg_pwr = (pwr.groupby(KEYS, dropna=False)["power"]
             .agg(mean="mean", se=lambda x: x.std(ddof=1)/np.sqrt(len(x))).reset_index())
agg_fdr = (fdr.groupby(KEYS, dropna=False)["fdr"]
             .agg(mean="mean", se=lambda x: x.std(ddof=1)/np.sqrt(len(x))).reset_index())
print("\nagg_fdr built:", agg_fdr.shape)
display(agg_fdr.head())

grouping by: ['cov_kind', 'n1', 'n2', 'rho', 'alpha', 'method', 'vm']

agg_fdr built: (108, 9)


,cov_kind,n1,n2,rho,alpha,method,vm,mean,se
0,block_ar1,80,80,0.6,0.05,BH,NaN,0.061944,0.004473
1,block_ar1,80,80,0.6,0.05,BY,NaN,0.007448,0.002139
2,block_ar1,80,80,0.6,0.05,LCT-B,NaN,0.049740,0.003157
3,block_ar1,80,80,0.6,0.05,LCT-N,cai_liu,0.106344,0.003816
4,block_ar1,80,80,0.6,0.05,LCT-N,gaussian,0.091647,0.003751


In [4]:
# how many reps per combination (completeness check)
b = df[df.cov_kind=="block_ar1"]
print("reps per (n1,n2):")
print(b.groupby(["n1","n2"]).size().to_string())
print("rho values:", sorted(b.rho.unique()), "\n")

# LCT-N vs LCT-B vs BH FDR under block_ar1, across matched n, alpha=0.05
q = agg_fdr[(agg_fdr.cov_kind=="block_ar1") &
            (agg_fdr.alpha=="0.05") &
            (agg_fdr.n1==agg_fdr.n2)]
print(f"{'n':>5} {'method':>8} {'vm':>10} {'FDR':>7} {'SE':>7}")
for _, r in q[q.method.isin(["LCT-N","LCT-B","BH"])].sort_values(["n1","method"]).iterrows():
    print(f"{r.n1:>5.0f} {r.method:>8} {str(r.vm):>10} {r['mean']:>7.3f} {r['se']:>7.4f}")

reps per (n1,n2):
n1   n2 
80   80     320
     120    320
     200    320
120  80     320
     120    320
     200    320
200  80     320
     120    320
     200    320
rho values: [np.float64(0.6)] 

    n   method         vm     FDR      SE
   80       BH        nan   0.062  0.0045
   80    LCT-B        nan   0.050  0.0032
   80    LCT-N    cai_liu   0.106  0.0038
   80    LCT-N   gaussian   0.092  0.0038
   80    LCT-N  jackknife   0.043  0.0066
  120       BH        nan   0.052  0.0027
  120    LCT-B        nan   0.045  0.0021
  120    LCT-N    cai_liu   0.062  0.0025
  120    LCT-N   gaussian   0.055  0.0024
  120    LCT-N  jackknife   0.051  0.0036
  200       BH        nan   0.055  0.0026
  200    LCT-B        nan   0.051  0.0024
  200    LCT-N    cai_liu   0.054  0.0024
  200    LCT-N   gaussian   0.051  0.0024
  200    LCT-N  jackknife   0.053  0.0026
